<a href="https://colab.research.google.com/github/padhisneha2025-dev/python30/blob/main/Day_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
pip install aiohttp beautifulsoup4


In [6]:
#Web Scraper -> CSV Export

import asyncio
import aiohttp
from bs4 import BeautifulSoup
from collections import deque
import csv
import json
import logging
import time
import re
from datetime import datetime
from dataclasses import dataclass, asdict
from typing import Optional

In [7]:
pip install datetime dataclasses optional

In [8]:
import logging

# Logging Setup

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("scrapper.log"),
        logging.StreamHandler()
    ]
)
log = logging.getLogger(__name__)

In [9]:
# ── DATA MODEL ────────────────────────────────────────
@dataclass
class Book:
    title: str
    price: float
    rating: str
    availability: str
    url: str
    scraped_at: str

# ── HELPERS ───────────────────────────────────────────
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

def parse_price(price_str: str) -> float:
    """Clean and convert price string to float"""
    cleaned = re.sub(r"[^\d.]", "", price_str)
    try:
        return float(cleaned)
    except ValueError:
        return 0.0

def parse_rating(article) -> str:
    """Extract star rating from class name"""
    rating_tag = article.find("p", class_="star-rating")
    if rating_tag:
        word = rating_tag["class"][1]
        return str(RATING_MAP.get(word, 0))
    return "0"

def parse_availability(article) -> str:
    avail = article.find("p", class_="availability")
    return avail.text.strip() if avail else "Unknown"

In [10]:
# ── ASYNC FETCHER ─────────────────────────────────────
async def fetch_page(session: aiohttp.ClientSession, url: str) -> Optional[str]:
    """Async fetch a single page"""
    try:
        async with session.get(url, timeout=aiohttp.ClientTimeout(total=10)) as resp:
            if resp.status == 200:
                return await resp.text()
            log.warning(f"Status {resp.status} for {url}")
            return None
    except Exception as e:
        log.error(f"Failed to fetch {url}: {e}")
        return None

In [11]:
# ── PARSER ────────────────────────────────────────────
def parse_books(html: str, page_url: str) -> tuple[list[Book], list[str]]:
    """Parse books and next page link from HTML"""
    soup = BeautifulSoup(html, "html.parser")
    books = []
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    for article in soup.find_all("article", class_="product_pod"):
        try:
            title = article.h3.a["title"]
            price = parse_price(article.find("p", class_="price_color").text)
            rating = parse_rating(article)
            availability = parse_availability(article)

            books.append(Book(
                title=title,
                price=price,
                rating=rating,
                availability=availability,
                url=page_url,
                scraped_at=now
            ))
        except Exception as e:
            log.warning(f"Failed to parse book: {e}")

    # get next page link
    links = []
    next_btn = soup.find("li", class_="next")
    if next_btn:
        next_href = next_btn.a["href"]
        base = "https://books.toscrape.com/catalogue/"
        if "catalogue" not in next_href:
            next_href = base + next_href
        else:
            next_href = "https://books.toscrape.com/" + next_href
        links.append(next_href)

    return books, links

In [12]:
# ── BFS ASYNC CRAWLER ─────────────────────────────────
async def crawl(start_url: str, max_pages: int) -> list[Book]:
    """BFS async crawler using queue + visited set"""
    queue = deque([start_url])
    visited = {start_url}          # hash set — O(1) dedup
    all_books = []
    pages_done = 0

    # semaphore — max 3 concurrent requests (polite crawling)
    semaphore = asyncio.Semaphore(3)

    async def fetch_with_semaphore(session, url):
        async with semaphore:
            await asyncio.sleep(0.3)   # polite delay
            return await fetch_page(session, url)

    async with aiohttp.ClientSession(headers={
        "User-Agent": "Mozilla/5.0 (research scraper)"
    }) as session:

        while queue and pages_done < max_pages:
            # batch — process up to 3 pages at once
            batch = []
            while queue and len(batch) < 3:
                batch.append(queue.popleft())

            # fetch batch concurrently
            tasks = [fetch_with_semaphore(session, url) for url in batch]
            results = await asyncio.gather(*tasks)

            for url, html in zip(batch, results):
                if not html:
                    continue

                books, links = parse_books(html, url)
                all_books.extend(books)
                pages_done += 1

                log.info(f"[{pages_done}/{max_pages}] {url} → {len(books)} books")

                for link in links:
                    if link not in visited:
                        visited.add(link)
                        queue.append(link)

    return all_books

In [13]:
# ── DEDUPLICATION ─────────────────────────────────────
def deduplicate(books: list[Book]) -> list[Book]:
    """Remove duplicate books by title using hash map"""
    seen = {}
    for book in books:
        if book.title not in seen:
            seen[book.title] = book
    original = len(books)
    unique = list(seen.values())
    log.info(f"Deduplication: {original} → {len(unique)} books")
    return unique

In [14]:
# ── EXPORT ────────────────────────────────────────────
def export_csv(books: list[Book], filename="books.csv"):
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=Book.__dataclass_fields__.keys())
        writer.writeheader()
        writer.writerows([asdict(b) for b in books])
    log.info(f"CSV saved → {filename}")

def export_json(books: list[Book], filename="books.json"):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump([asdict(b) for b in books], f, indent=2)
    log.info(f"JSON saved → {filename}")

In [15]:
# ── ANALYTICS ─────────────────────────────────────────
def print_analytics(books: list[Book]):
    if not books:
        return

    prices = [b.price for b in books if b.price > 0]
    avg_price = sum(prices) / len(prices) if prices else 0
    top_rated = sorted(books, key=lambda b: b.rating, reverse=True)[:5]

    print("\n" + "─" * 50)
    print("📊 SCRAPE ANALYTICS")
    print("─" * 50)
    print(f"  Total books scraped : {len(books)}")
    print(f"  Average price       : £{avg_price:.2f}")
    print(f"  Cheapest book       : £{min(prices):.2f}")
    print(f"  Most expensive      : £{max(prices):.2f}")
    print(f"\n  ⭐ Top 5 Rated:")
    for b in top_rated:
        print(f"    {b.rating}★ — {b.title[:50]} — £{b.price}")
    print("─" * 50 + "\n")

# ── MAIN ──────────────────────────────────────────────
async def main():
    START_URL = "https://books.toscrape.com/catalogue/page-1.html"
    MAX_PAGES = 20

    print("\n🕷️  Advanced Async Book Scraper")
    print(f"   Target  : {START_URL}")
    print(f"   Max pages: {MAX_PAGES}\n")

    start = time.time()
    books = await crawl(START_URL, MAX_PAGES)
    books = deduplicate(books)
    elapsed = time.time() - start

    export_csv(books)
    export_json(books)
    print_analytics(books)

    print(f"  ⏱️  Completed in {elapsed:.2f}s")

In [16]:
# ── RUN ───────────────────────────────────────────────
await main()


🕷️  Advanced Async Book Scraper
   Target  : https://books.toscrape.com/catalogue/page-1.html
   Max pages: 20


──────────────────────────────────────────────────
📊 SCRAPE ANALYTICS
──────────────────────────────────────────────────
  Total books scraped : 399
  Average price       : £34.97
  Cheapest book       : £10.16
  Most expensive      : £59.90

  ⭐ Top 5 Rated:
    5★ — Sapiens: A Brief History of Humankind — £54.23
    5★ — Set Me Free — £17.46
    5★ — Scott Pilgrim's Precious Little Life (Scott Pilgri — £52.29
    5★ — Rip it Up and Start Again — £35.02
    5★ — Chase Me (Paris Nights #2) — £25.27
──────────────────────────────────────────────────

  ⏱️  Completed in 7.50s


In [17]:
#Web Scraper -> CSV Export

import asyncio
import aiohttp
from bs4 import BeautifulSoup
from collections import deque
import csv
import json
import logging
import time
import re
from datetime import datetime
from dataclasses import dataclass, asdict
from typing import Optional